# 04: Model comparison

Milestone 3. Learned models are tuned and compared against the Milestone 2 baselines on the same
rows, features, folds and metrics, all from `src/chess_blunder/modelling.py`.

The rules carried over from `03_preprocessing_baseline.ipynb`:

- Every setting is chosen by **mean PR-AUC over the five grouped folds on the training games**.
- The test set (the most recent 20% of games) is not used to choose anything. It is scored once,
  for the chosen models only, later in this notebook.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

from chess_blunder import config, modelling

pd.set_option('display.width', 120)
FIG_DIR = config.REPORTS_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

positions = pd.read_parquet(config.PROCESSED_DIR / 'positions.parquet')

## 1. Same rows, same split, same folds

Rebuilt with the Milestone 2 functions and checked again, so this notebook stands on its own.

In [2]:
table = modelling.modelling_table(positions)
is_test = modelling.time_split(table)
train, test = table[~is_test].reset_index(drop=True), table[is_test].reset_index(drop=True)
folds = modelling.cv_folds(train)

assert not set(train.game_id) & set(test.game_id), 'a game is on both sides of the split'
assert train.end_time.max() < test.end_time.min(), 'test games are not all later than training games'
for fit_idx, val_idx in folds:
    assert not set(train.game_id.iloc[fit_idx]) & set(train.game_id.iloc[val_idx]), 'game split across folds'

print(f'train: {len(train):,} moves in {train.game_id.nunique():,} games, blunder rate {train[modelling.TARGET].mean():.2%}')
print(f'test : {len(test):,} moves in {test.game_id.nunique():,} games (not used until section 5)')

train: 102,880 moves in 3,206 games, blunder rate 9.39%
test : 27,168 moves in 802 games (not used until section 5)


## 2. Baselines

Recomputed on the same folds rather than read from `baselines_cv.csv`, so their out-of-fold
predictions are available for the plots later.

In [3]:
cv_scores, oof = {}, {}
for name, model in {'base rate': modelling.BaseRate(), 'stage x complexity': modelling.StageComplexityRate()}.items():
    cv_scores[name], oof[name] = modelling.out_of_fold(model, train, folds)

pd.DataFrame({name: s.drop(columns='base_rate').mean() for name, s in cv_scores.items()}).T.round(4)

,pr_auc,roc_auc,brier
base rate,0.0939,0.5000,0.0851
stage x complexity,0.1320,0.6383,0.0833


## 3. Logistic regression

The simplest learned model: a weighted sum of the features passed through a sigmoid. Three choices:

- **Splines on `cp_before` and `win_before`.** A linear model can only say "more of this feature,
  more (or less) risk". The table below shows the engine evaluation does not behave like that.

In [4]:
bands = pd.cut(train.win_before, np.arange(0, 101, 10))
train.groupby(bands, observed=True)[modelling.TARGET].agg(['mean', 'size']).rename(
    columns={'mean': 'blunder rate', 'size': 'moves'}).round(3)

,blunder rate,moves
win_before,,
"(0, 10]",0.006,18887
"(10, 20]",0.017,6516
"(20, 30]",0.140,3386
"(30, 40]",0.227,4106
"(40, 50]",0.128,16128
"(50, 60]",0.073,18129
"(60, 70]",0.180,5529
"(70, 80]",0.263,3942
"(80, 90]",0.207,6553


Risk is under 2% when I am clearly losing (`win_before` below 20): a 15-point drop in winning
chances is almost impossible from there, so this is partly how the label works rather than how I
play. It is 6.5% when I am clearly winning (above 90). It peaks at 20–26% where one side is ahead
but the game is not decided (30–40 and 70–80), and dips at dead level (40–60). Two peaks and a dip
cannot be fitted by one weight, so both engine columns are expanded into cubic splines with knots
at quantiles.

- **No class weights.** Blunders are ~9% of moves. Reweighting them would make the model rank moves
  much the same way but push every predicted risk upwards, which spoils the Brier score and the
  reliability plot. PR-AUC measures ranking, so it loses nothing.
- **Tuning the regularisation strength `C`** over 0.01, 0.1, 1 and 10. Smaller `C` shrinks the
  weights towards zero; larger `C` lets the model fit the training folds more closely.

In [5]:
lr = modelling.tune(modelling.logistic_regression, {'C': [0.01, 0.1, 1, 10]}, train, folds)
cv_scores['logistic regression'] = lr.results
oof['logistic regression'] = lr.oof

assert len(lr.oof) == len(train), 'out-of-fold predictions must cover every training row'
print('chosen:', lr.best_params)
lr.results.round(4)

chosen: {'C': 10}


,C,mean pr_auc,std pr_auc,mean roc_auc,std roc_auc,mean brier,std brier
0,0.01,0.2294,0.0109,0.7673,0.0069,0.0785,0.0015
1,0.10,0.2348,0.0106,0.7801,0.0079,0.0778,0.0014
2,1.00,0.2350,0.0104,0.7825,0.0078,0.0777,0.0015
3,10.00,0.2352,0.0103,0.7827,0.0079,0.0777,0.0015


`C` hardly matters. From 0.1 to 10, mean PR-AUC moves by 0.0004, about a twenty-fifth of the
fold std (0.010). With ~100,000 training moves and 28 columns after the splines and one-hot
encoding, there is little to overfit, so regularisation has almost nothing to do. `C = 10` wins on
the rule and sits at the edge of the grid, but extending the grid would only chase noise. Only
`C = 0.01` is visibly worse: it shrinks the weights too far.

### Do the splines earn their place?

The same model at the chosen `C`, with the engine columns left linear. If the splines matter, this
should score clearly lower.

In [6]:
linear = make_pipeline(modelling.make_preprocessor(), LogisticRegression(C=lr.best_params['C'], max_iter=1000))
linear_scores = modelling.cross_validate(linear, train, folds).drop(columns='base_rate')
spline_row = lr.results.loc[lr.results.C == lr.best_params['C']].iloc[0]

pd.DataFrame({
    'with splines': {m: spline_row[f'mean {m}'] for m in ('pr_auc', 'roc_auc', 'brier')},
    'linear only': linear_scores.mean(),
}).T.round(4)

,pr_auc,roc_auc,brier
with splines,0.2352,0.7827,0.0777
linear only,0.1561,0.6726,0.0828


They do. Without them, PR-AUC falls from 0.235 to 0.156, only a little above the stage ×
complexity table (0.132). Most of what logistic regression adds over the baseline comes from
reading the shape of the engine evaluation. That also means part of the lift is the label effect
above: decided positions rarely produce blunders, and the model learns that quickly. The error
analysis later in this notebook slices by `win_before` band to show how much is left beyond it.

## Results so far

Cross-validated on the training games, mean ± std over five folds. Each learned model is shown at
its chosen setting.

In [7]:
def summarise(scores: pd.DataFrame) -> dict:
    """Mean ± std of each metric: from per-fold scores, or from a tuning row already aggregated."""
    if 'mean pr_auc' in scores:
        row = scores.loc[scores['mean pr_auc'].idxmax()]
        return {m: f"{row[f'mean {m}']:.3f} ± {row[f'std {m}']:.3f}" for m in ('pr_auc', 'roc_auc', 'brier')}
    return {m: f'{scores[m].mean():.3f} ± {scores[m].std():.3f}' for m in ('pr_auc', 'roc_auc', 'brier')}

comparison = pd.DataFrame({name: summarise(s) for name, s in cv_scores.items()}).T
comparison.to_csv(config.REPORTS_DIR / 'model_comparison_cv.csv')
comparison

,pr_auc,roc_auc,brier
base rate,0.094 ± 0.002,0.500 ± 0.000,0.085 ± 0.002
stage x complexity,0.132 ± 0.003,0.638 ± 0.004,0.083 ± 0.002
logistic regression,0.235 ± 0.010,0.783 ± 0.008,0.078 ± 0.001


Logistic regression lifts mean PR-AUC from 0.132 to 0.235, two and a half times the base rate,
and lowers the Brier score from 0.083 to 0.078. The gap to the stage × complexity table is about
ten fold stds, so it is not noise. Next: gradient boosting and a random forest, tuned through the
same `tune` function on the same folds.